# Intro to Skill Registry

> following [this notebook example](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/skill-registry/intro_skill_registry.ipynb)

## Overview
The Skill Registry is a secure, private repository for storing, indexing, and dynamically discovering agent skills on the Gemini Enterprise Agent Platform.

A skill extends an agent's capabilities by providing precise system instructions, domain-specific documentation, and local or remote tools. By indexing skills in the Registry, agents can perform semantic search at runtime to dynamically discover and load the exact capabilities needed to solve a user's request, optimizing context window usage and enforcing robust access control.

## Objectives
In this notebook, you will learn how to:

1. Define a Custom Skill: Create a local skill package structure with instructions and metadata.
2. Upload and Register a Skill: Programmatically register custom skills in the Skill Registry.
3. Ingest Skills from GitHub: Batch-import ready-to-use agent skills directly from a GitHub repository.
4. Perform Semantic Discovery: Retrieve and search relevant skills dynamically using semantic query matching.

## Getting Started

### Install libraries

In [ ]:
# %pip install --upgrade --quiet "google-cloud-aiplatform==1.152.0"
# print(aiplatform.__version__)

In [1]:
import os
import re
import sys
import shutil
import tempfile
import urllib.request
import yaml
import zipfile

from dotenv import load_dotenv
from datetime import datetime

# import vertexai
# print(f"vertexai version: {vertexai.__version__}")

# import google.cloud.aiplatform as aiplatform
# print(f"aiplatform version: {aiplatform.__version__}")

import agentplatform
# from google.genai import agentplatform
print(f"agentplatform version: {agentplatform.__version__}")

agentplatform version: 1.163.0


In [2]:
try:
    # Works when running as a standard script
    base_dir = os.path.dirname(__file__)
except NameError:
    # Fallback when running inside a Jupyter/Colab notebook
    base_dir = os.getcwd()

PROJECT_ROOT = os.path.abspath(os.path.join(base_dir, ".."))

# # Ensure the project root is on sys.path so `src` is importable
# PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
# sys.path.insert(0, PROJECT_ROOT)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

PROJECT_ROOT: /home/user/geap/geap-tour


In [3]:
ENV_FILE_PATH = os.path.join(PROJECT_ROOT, ".env")
load_dotenv(dotenv_path=ENV_FILE_PATH)

print(f"ENV_FILE_PATH: {ENV_FILE_PATH}")

ENV_FILE_PATH: /home/user/geap/geap-tour/.env


In [4]:
# fmt: off
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.getenv("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.getenv("GCP_REGION")

print(f"PROJECT_ID: {PROJECT_ID}")
print(f"LOCATION: {LOCATION}")

PROJECT_ID: hybrid-vertex
LOCATION: us-central1


### Initialize the Agent Platform client

In [5]:
# vertexai.init(project=PROJECT_ID, location=LOCATION)
# client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

# from google.genai import agentplatform
# import agentplatform
client = agentplatform.Client(project=PROJECT_ID, location=LOCATION)

## Define and Register a Custom Skill

To register a skill in the Skill Registry, you need a skill package. At its simplest, a skill package is a local directory containing a mandatory SKILL.md file.

The SKILL.md file follows a specific layout:

1. **YAML Frontmatter**: A metadata block at the top specifying:
* `name`: A unique identifier matching the skill package name.
* `description`: A capability statement (starting in third-person) explaining what the skill does.
2. **Instructions**: The core markdown instructions that guide the agent's behavior when executing the skill.

### Create a sample skill package
The following cell programmatically creates a temporary directory (`/tmp/sample_math_skill`) containing a sample `SKILL.md` file for a simple math helper.

In [6]:
# Create a temporary directory to simulate a local skill package
local_skill_dir = "/tmp/sample_math_skill"
os.makedirs(local_skill_dir, exist_ok=True)

# Create the SKILL.md file which defines the skill
skill_md_content = """---
name: sample-math-skill
description: A sample skill that helps with simple math addition. Use this skill when you need to add two numbers.
---
# Test Math Skill
This skill provides instructions on how to add two numbers.

## Instructions
To add two numbers, sum them up. For example, 2 + 2 = 4.
"""

with open(os.path.join(local_skill_dir, "SKILL.md"), "w") as f:
    f.write(skill_md_content)

print(f"Created sample skill directory at: {local_skill_dir}")

Created sample skill directory at: /tmp/sample_math_skill


### Upload and register the skill package
Now, upload and register your custom skill in the Skill Registry using client.skills.create.

* The SDK will automatically package, compress, and upload the local skill folder specified in "local_path".
* By default, the operation blocks synchronously (wait_for_completion=True) until the backend has successfully provisioned and indexed the skill.

In [8]:
ts = datetime.now().strftime("%Y%m%d-%H%m%S")
user_skill_id = f"math-skill-{ts}"

try:
    # Call create with local_path in config
    # By default, wait_for_completion is True, so this will block until the skill is created
    skill = client.skills.create(
        skill_id=user_skill_id,
        display_name="Sample math skill",
        description="This skill provides functions to perform math calculations",
        config={
            # "skill_id": user_skill_id,
            "local_path": "/tmp/sample_math_skill"
        }
        )
    print(f"SUCCESS: Skill created successfully!")
    print(f"Skill Name: {skill.name}")
    print(f"Skill State: {skill.state}")
    print(f"Skill Display Name: {skill.display_name}")
    print(f"Skill Description: {skill.description}")
except Exception as e:
    print(f"FAILED: Skill creation failed: {e}")
    skill = None

SUCCESS: Skill created successfully!
Skill Name: projects/934903580331/locations/us-central1/skills/math-skill-20260812-090817
Skill State: SkillState.ACTIVE
Skill Display Name: Sample math skill
Skill Description: This skill provides functions to perform math calculations


#### List skills

In [9]:
pager = client.skills.list()
for skill in pager:
    print(skill.name, skill.display_name)

projects/934903580331/locations/us-central1/skills/math-skill-20260812-090817 Sample math skill
projects/934903580331/locations/us-central1/skills/google-cloud-waf-security-20260812-085535 google-cloud-waf-security
projects/934903580331/locations/us-central1/skills/data-manager-api-audience-ingestion-20260812-085524 data-manager-api-audience-ingestion
projects/934903580331/locations/us-central1/skills/google-mobile-ads-interstitial-20260812-085514 google-mobile-ads-interstitial
projects/934903580331/locations/us-central1/skills/data-manager-api-event-ingestion-20260812-085503 data-manager-api-event-ingestion
projects/934903580331/locations/us-central1/skills/google-mobile-ads-banner-20260812-085453 google-mobile-ads-banner
projects/934903580331/locations/us-central1/skills/google-ads-api-quickstart-20260812-085442 google-ads-api-quickstart
projects/934903580331/locations/us-central1/skills/data-manager-api-setup-20260812-085432 data-manager-api-setup
projects/934903580331/locations/us-

## Batch Ingestion of Skills from a GitHub Repository

In real-world production environments, organization-wide agent skills are often stored and managed in a central Git repository.

This section demonstrates how to build an automated ingestion pipeline to fetch, parse, and batch-register skills from a remote repository.

### Remote repository - example #1

We will use the Google-managed [google/skills](https://github.com/google/skills) repository as our input. This repository contains ready-to-use agent skills, such as:

* **`bigquery-basics`**: Fundamental tools and instructions for querying Google Cloud BigQuery datasets.
* **`cloud-run-basics`**: Key operations and deployment guidelines for managing Google Cloud Run services.

In [ ]:
GITHUB_REPO_URL = "https://github.com/google/skills"

#### Define pipeline helper functions

Define the necessary Python helper functions to automate the ingestion workflow. This includes downloading the zip archive, extracting the files, scanning the directory tree, parsing frontmatter schemas, and registering each skill.

In [ ]:
class Skill:
  def __init__(self, name, description, local_skill_dir):
    self.name = name
    self.description = description
    self.local_skill_dir = local_skill_dir

  def __str__(self):
    return f"Skill(name={self.name}, description={self.description}, local_skill_dir={self.local_skill_dir})"

# def parse_skill_md(filepath: str) -> tuple[str, str]:
#   """Parses SKILL.md file to get the skill name and description."""
#   name = "Untitled Skill"
#   description = "No description provided."
#   try:
#     with open(filepath, "r", encoding="utf-8") as f:
#       content = f.read()
#       if content.startswith("---"):
#         end_idx = content.find("---", 3)
#         if end_idx != -1:
#           yaml_part = content[3:end_idx]
#           try:
#             data = yaml.safe_load(yaml_part)
#             if data:
#               name = data.get("name", name)
#               description = data.get("description", description)
#           except yaml.YAMLError as yaml_e:
#             print(f"YAML parsing warning (using fallback): {yaml_e}")
#             match_name = re.search(r"^name:\s*(.+)$", yaml_part, re.M)
#             if match_name:
#               name = match_name.group(1).strip()
#             match_desc = re.search(
#                 r"^description:\s*(?:>-\s*)?(.+)$", yaml_part, re.M
#             )
#             if match_desc:
#               description = match_desc.group(1).strip()
#   except IOError as e:
#     print(f"Failed to parse {filepath}: {e}")
#   return name, description

def parse_skill_md(filepath: str) -> tuple[str, str]:
    """Parses SKILL.md file to get the skill name and description."""
    name = "Untitled Skill"
    description = "No description provided."
    
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
            
        if content.startswith("---"):
            end_idx = content.find("---", 3)
            if end_idx != -1:
                yaml_part = content[3:end_idx]
                try:
                    data = yaml.safe_load(yaml_part)
                    if data:
                        # --- PROGRAMMATIC FIX FOR BACKEND VALIDATION ---
                        # Loop through metadata and flatten complex non-scalar elements
                        sanitized_data = {}
                        for key, value in data.items():
                            if isinstance(value, list):
                                # Convert lists to comma-separated strings
                                sanitized_data[key] = ", ".join(map(str, value))
                            elif isinstance(value, dict):
                                # Convert dictionaries to string representation or flatten them
                                sanitized_data[key] = str(value)
                            else:
                                sanitized_data[key] = value
                        
                        # (Optional) If you rewrite the file locally before creating the skill:
                        # new_yaml = yaml.safe_dump(sanitized_data)
                        # with open(filepath, "w", encoding="utf-8") as f:
                        #     f.write(f"---\n{new_yaml}---\n" + content[end_idx+3:])
                        
                        name = sanitized_data.get("name", name)
                        description = sanitized_data.get("description", description)
                        
                except yaml.YAMLError as yaml_e:
                    print(f"YAML parsing warning (using fallback): {yaml_e}")
                    # Fallback regex logic remains here...
                    
    except IOError as e:
        print(f"Failed to parse {filepath}: {e}")
        
    return name, description

def get_all_skills_from_dir(repo_dir: str) -> list[Skill]:
  """Returns a list of Skill objects from the given directory."""
  skills = []
  skills_found = 0
  seen_skills = set()
  for root, dirs, files in os.walk(repo_dir):
    lower_files = {f.lower(): f for f in files}
    if "skill.md" in lower_files:
      # Prune subdirectories to avoid deeper recursion in this skill folder.
      dirs[:] = []

      skill_md_filename = lower_files["skill.md"]
      filepath = os.path.join(root, skill_md_filename)

      name, description = parse_skill_md(filepath)

      # De-dupe based on skill name
      if name in seen_skills:
        print(f"Skipping duplicate skill name: {name}")
        continue
      seen_skills.add(name)

      skills.append(Skill(name, description, root))
      skills_found += 1
  print(f"Found {skills_found} skills.")
  return skills

def download_github_repo(repo_url, output_dir):
  os.makedirs(output_dir, exist_ok=True)
  branch = "master"

  zip_url = f"{GITHUB_REPO_URL}/archive/refs/heads/{branch}.zip"
  print(f"Attempting to download repository zip via HTTP: {zip_url}...")

  with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as temp_zip_file:
    zip_path = temp_zip_file.name

  request = urllib.request.Request(zip_url)
  with urllib.request.urlopen(request, timeout=30) as response:
    with open(zip_path, "wb") as out_file:
      shutil.copyfileobj(response, out_file)
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)
  print(
      f"Repository extracted successfully from '{branch}' branch zip"
      " archive."
  )
  os.remove(zip_path)
  print(f"Deleted temporary zip file: {zip_path}")

# def create_skill(skill):
#   name = skill.name
#   description = skill.description
#   skill_dir = skill.local_skill_dir
#   print(f"Creating skill: {skill}")
#   timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
#   skill = client.skills.create(
#       display_name=name,
#       description=description,
#       config={
#           "local_path": skill_dir,
#           "skill_id": f"{name}-{timestamp}"
#       }
#   )
#   print(f"Created skill: {name}")

def create_skill(skill):
    name = skill.name
    description = skill.description
    skill_dir = skill.local_skill_dir
    print(f"Creating skill: {skill}")
    
    # 1. Clean the skill name to meet requirements
    # Convert to lowercase
    sanitized_name = name.lower()
    # Replace any sequence of non-alphanumeric characters with a single hyphen
    sanitized_name = re.sub(r'[^a-z0-9]+', '-', sanitized_name)
    # Strip any leading or trailing hyphens to ensure it starts/ends properly
    sanitized_name = sanitized_name.strip('-')
    
    # If the parsed name somehow becomes empty, provide a valid fallback letter
    if not sanitized_name:
        sanitized_name = "skill"
        
    # 2. Format the timestamp
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    
    # 3. Combine them and slice to ensure it stays strictly under 63 characters
    skill_id = f"{sanitized_name}-{timestamp}"
    if len(skill_id) > 63:
        # Keep the timestamp intact at the end, truncate the prefix
        allowed_prefix_len = 63 - len(timestamp) - 1
        skill_id = f"{sanitized_name[:allowed_prefix_len].strip('-')}-{timestamp}"

    # Verify final character boundary check
    print(f"Generated compliant Skill ID: {skill_id}")

    # 4. Invoke the Client Call
    skill_response = client.skills.create(
        display_name=name,  # display_name can keep spaces/capitalization
        description=description,
        config={
            "local_path": skill_dir,
            "skill_id": skill_id
        }
    )
    print(f"Created skill: {name}")
    return skill_response

### Run the batch ingestion pipeline

Specify your target repository URL, trigger the download, and execute the bulk skill ingestion. Each discovered skill will be individually registered with a unique timestamped identifier.

In [ ]:
output_dir = tempfile.mkdtemp()
download_github_repo(GITHUB_REPO_URL, output_dir)

# skills = get_all_skills_from_dir(output_dir)
# for skill in skills:
#     create_skill(skill)

In [ ]:
# 1. Fetch your list of identified skills
all_skills = get_all_skills_from_dir(output_dir)

# 2. Initialize a creation counter
created_count = 0
MAX_SKILLS = 15
print(f"Starting skill deployment process. Limit set to {MAX_SKILLS} skills.")

# 3. Iterate and enforce the condition
for skill in all_skills:
    # Check the condition BEFORE attempting to create the next skill
    if created_count >= MAX_SKILLS:
        print(f"Reached the maximum limit of {MAX_SKILLS} created skills. Stopping.")
        break
        
    try:
        # Attempt to upload and create the skill via the API
        create_skill(skill)
        created_count += 1
        print(f"Progress: {created_count}/{MAX_SKILLS} skills created.")
        
    except Exception as e:
        # Catch validation or API errors so one bad skill doesn't crash the whole process
        print(f"Skipping skill '{skill.name}' due to error: {e}")

print(f"Process complete. Total skills created in this run: {created_count}")

### Remote repository - example #2

Optionally, batch ingest from a repository of your own, e.g., my [tottenjordan/me-skittles](https://github.com/tottenjordan/me-skittles) repository.

In [ ]:
# GITHUB_REPO_URL_v2 = "https://github.com/tottenjordan/me-skittles"

In [ ]:
# output_dir = tempfile.mkdtemp()
# download_github_repo(GITHUB_REPO_URL_v2, output_dir)

# skills = get_all_skills_from_dir(output_dir)
# for skill in skills:
#     create_skill(skill)

## Semantic Skill Discovery and Retrieval

The Skill Registry has built-in vector search capabilities. When skills are registered, their metadata descriptions are automatically vectorized and indexed.

This allows you to perform **semantic search** using natural language queries (e.g., searching for "firebase" or "database helpers") to locate the most relevant skills, even if there are no exact keyword matches.

Agents use this exact retrieval mechanism at runtime to dynamically discover and load matching skills based on the user's intent.

In [10]:
print(f"\n--- Search Relevant Skills ---")
query = "firebase"
print(f"Searching for skills matching query: '{query}'...")
try:
    # Call retrieve to perform semantic search
    response = client.skills.retrieve(
        query=query,
        config={"top_k": 2}
    )
    print(f"SUCCESS: Skills retrieved successfully!")
    print(f"Found {len(response.retrieved_skills)} matching skills.")
    for i, retrieved in enumerate(response.retrieved_skills):
        print(f"  [{i+1}] Skill Name: {retrieved.skill_name}")
        print(f"  [{i+1}] Description: {retrieved.description}")
except Exception as e:
    print(f"FAILED: Retrieve skills failed: {e}")


--- Search Relevant Skills ---
Searching for skills matching query: 'firebase'...
SUCCESS: Skills retrieved successfully!
Found 2 matching skills.
  [1] Skill Name: projects/934903580331/locations/us-central1/skills/firebase-basics-20260812-083209
  [1] Description: Provides foundational Firebase CLI setup, CLI installation, version checks (`firebase-tools@latest --version`), CLI login (including --no-localhost), project creation, project selection (`firebase use`), and app config file downloads (`google-services.json`, `GoogleService-Info.plist`). Use ONLY for CLI login, project creation/switching, or downloading app config files. Don't use for Firebase Hosting deploy, Firestore, Auth, App Hosting, Data Connect, Crashlytics, or Remote Config.
  [2] Skill Name: projects/934903580331/locations/us-central1/skills/firebase-basics-20260812-084227
  [2] Description: Provides foundational Firebase CLI setup, CLI installation, version checks (`firebase-tools@latest --version`), CLI login (in